# Stage 9 — Paper tables and hostile-reviewer consistency check

This notebook turns the evidence of Stages 0–8 into the manuscript files and then attacks them like a hostile reviewer.

* **Inputs (read only):** `reports/tables/*.csv`, `data/processed/*.parquet` (row counts only), the frozen test predictions in
  `models/test/` (read, never written), `reports/paper_draft.md`, `reports/references.bib`, and the first draft
  `reports/drafts/paper_draft_v0.md` (a snapshot taken before the review, kept so the review is reproducible).
* **Outputs:** `reports/tables/09_*.csv|tex` (derived numbers, paper Tables 1–4, TFT test coverage, review issues, number
  cross-check), Tables 1–4 inserted into `reports/paper_draft.md`, and `reports/paper_draft.tex`.
* **Rules that matter here:** every number in the Abstract and Results must come from a CSV (rule 10) and is linked to it by a
  machine-readable source comment; significance may only be claimed from the Holm-corrected Diebold–Mariano tests; nothing in
  `models/test/`, `manifest.json` or `final_config.yaml` is modified (rule 4, Stage-7 integrity).

The checker lives in `src/paper.py` (unit tests: `tests/test_paper.py`). A source comment looks like
`<!-- source: 07_test_cls_bootstrap.csv | index=catboost | BSS=0.376 -->`: the checker finds the row, rounds the CSV value to
the decimals written, and also verifies that `0.376` really appears in the sentence before the comment.

**WHAT.** Load libraries, paths and the checker module; print the pandas and pandoc versions.

**WHY.** Rule 15 (no hidden state): every path is explicit and the notebook only reads and writes files on disk.

**Alternatives considered.** Calling pandoc through `pypandoc`; not needed, `subprocess` is enough and adds no dependency.

**What to look for.** The versions match `requirements.txt` (pandas 3.0.5) and pandoc is found.

In [1]:
import sys, json, re, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import paper as P

TAB = ROOT / "reports" / "tables"
DRAFT, DRAFT_V0 = ROOT / "reports" / "paper_draft.md", ROOT / "reports" / "drafts" / "paper_draft_v0.md"
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)
np.random.seed(0)
print("pandas", pd.__version__, "| pandoc", subprocess.run(["pandoc", "--version"], capture_output=True, text=True).stdout.split()[1])

pandas 3.0.5 | pandoc 3.1.3


**Interpretation.** pandas 3.0.5 matches `requirements.txt`; pandoc 3.1.3 is a system tool (not a Python package), so it is named in the Reproducibility section instead.

**Decision.** Proceed; the notebook reads only files on disk.

**Paper note.** The manuscript was generated from version-controlled tables and converted to LaTeX with pandoc 3.1.3.

**WHAT.** Compute every *derived* number the manuscript needs (sums, ranges, counts, Spearman correlations) from the existing CSVs and store them with their source and method in `09_derived_numbers.csv`.

**WHY.** Some statements in the paper (e.g. 'ΔBSS +0.070 to +0.108', 'onsets caught 41–59 %') are not single cells of a table. Writing them down as a table makes them checkable like every other number, instead of being typed by hand.

**Alternatives considered.** Computing the ranges inside the manuscript checker (hides the method); recomputing from raw predictions (duplicates Stage 7–8 code and risks new bugs).

**What to look for.** Plausible values (test days 2,192; test wet share ≈ 0.557; 118 ablation contrasts). Any value that differs from an earlier write-up must be explained, not silently used.

In [2]:
rows = []
def add(q, v, source, method):
    rows.append({"quantity": q, "value": float(v), "source": source, "method": method})

# test period (target years 2020-2025)
yr = pd.read_csv(TAB / "08_errors_by_year.csv")
days = yr.year.map(lambda y: 366 if pd.Timestamp(f"{y}-12-31").dayofyear == 366 else 365)
n_test = int(pd.read_csv(TAB / "07_test_by_season_cls.csv").query("model == 'baseline:climatology'").n_days.sum())
assert days.sum() == n_test
add("test_days", n_test, "07_test_by_season_cls.csv", "sum of season day counts")
add("test_wet_share", (yr.wet_share * days).sum() / n_test, "08_errors_by_year.csv", "day-weighted mean of yearly wet shares")
add("test_mean_mm", yr.obs_total_mm.sum() / n_test, "08_errors_by_year.csv", "sum of yearly totals / test days")
add("test_max_mm", yr.max_day_mm.max(), "08_errors_by_year.csv", "max of yearly maxima")

# inhomogeneity of the target
ann = pd.read_csv(TAB / "00_annual_precip_totals.csv", index_col=0)
for a, b in [(1981, 1997), (1998, 2015), (2016, 2025)]:
    add(f"prectotcorr_mean_{a}_{b}", ann.loc[a:b, "PRECTOTCORR"].mean(), "00_annual_precip_totals.csv", "mean annual total")
for a, b in [(1998, 2015), (2016, 2025)]:
    add(f"imerg_mean_{a}_{b}", ann.loc[a:b, "IMERG_PRECTOT"].mean(), "00_annual_precip_totals.csv", "mean annual total")

# data set size (row counts only; no test labels are read)
feat = pd.read_parquet(ROOT / "data" / "processed" / "features_B.parquet")
fA = pd.read_parquet(ROOT / "data" / "processed" / "features_A.parquet")
add("n_rows", len(feat), "data/processed/features_B.parquet", "rows after warm-up")
add("n_features_A", sum(not c.startswith("y_") for c in fA), "data/processed/features_A.parquet", "non-target columns")
add("n_features_B", sum(not c.startswith("y_") for c in feat), "data/processed/features_B.parquet", "non-target columns")
add("n_dev_rows", (feat.index <= "2019-12-30").sum(), "data/processed/features_B.parquet", "feature date <= 2019-12-30")

# test comparisons (learners = classical + TSMixer + TFT; TiDE separately)
cmp = pd.read_csv(TAB / "07_statistical_comparison.csv")
learn = cmp[~cmp.model.str.startswith("baseline") & (cmp.model != "deep:tide")]
a_cls = learn.query("family == 'A' and task == 'cls'")
add("famA_cls_learner_dBSS_min", a_cls.estimate.min(), "07_statistical_comparison.csv", "min over learners")
add("famA_cls_learner_dBSS_max", a_cls.estimate.max(), "07_statistical_comparison.csv", "max over learners")
add("famA_cls_learner_p_holm_max", a_cls.p_holm.max(), "07_statistical_comparison.csv", "max over learners")
add("famA_cls_learners_significant", a_cls.significant.sum(), "07_statistical_comparison.csv", "count")
b_cls = cmp.query("family == 'B' and task == 'cls' and not model.str.startswith('baseline') and model != 'deep:tide'", engine="python")
add("famB_cls_learner_p_holm_min", b_cls.p_holm.min(), "07_statistical_comparison.csv", "min over learners")
reg_mae = cmp.query("family == 'A' and loss == 'abs_error' and not model.str.startswith('baseline')", engine="python")
add("famA_mae_better_significant", (reg_mae.significant & reg_mae.better).sum(), "07_statistical_comparison.csv", "count")
add("famA_mae_worse_significant", (reg_mae.significant & ~reg_mae.better).sum(), "07_statistical_comparison.csv", "count")

# seasonality of skill (learners = non-baseline classifiers in the table)
mon = pd.read_csv(TAB / "08_errors_by_month.csv").query("task == 'cls' and not model.str.startswith('baseline')", engine="python")
for name, months in [("transition", [3, 4, 10, 11]), ("monsoon", [6, 7, 8, 9])]:
    b = mon[mon.month.isin(months)].BSS
    add(f"{name}_month_bss_min", b.min(), "08_errors_by_month.csv", f"min BSS, learners, months {months}")
    add(f"{name}_month_bss_max", b.max(), "08_errors_by_month.csv", f"max BSS, learners, months {months}")

# development -> test
dvt = pd.read_csv(TAB / "07_dev_vs_test_ranking.csv")
for metric in ["BSS", "MAE", "MSESS"]:
    d = dvt[dvt.metric == metric]
    add(f"spearman_dev_test_{metric}", d["development (CV 2005-2019)"].corr(d["test (2020-2025)"], method="spearman"),
        "07_dev_vs_test_ranking.csv", "Spearman over models")
d = dvt.query("metric == 'BSS' and not model.str.startswith('baseline')", engine="python")["test - dev"]
add("test_minus_dev_BSS_learner_min", d.min(), "07_dev_vs_test_ranking.csv", "min over classical learners")
add("test_minus_dev_BSS_learner_max", d.max(), "07_dev_vs_test_ranking.csv", "max over classical learners")
d = dvt.query("metric == 'MAE' and model != 'baseline:majority'")["test - dev"]
add("test_minus_dev_MAE_min", d.min(), "07_dev_vs_test_ranking.csv", "min over models except majority")
add("test_minus_dev_MAE_max", d.max(), "07_dev_vs_test_ranking.csv", "max over models except majority")

# error structure
tr = pd.read_csv(TAB / "08_transitions.csv").query("not model.str.startswith('baseline')", engine="python")
for name, sub in [("onset", "dry today (onset?)"), ("cessation", "wet today (cessation?)")]:
    c = tr[tr.subset == sub].transitions_caught
    add(f"{name}_caught_min", c.min(), "08_transitions.csv", "min over learners")
    add(f"{name}_caught_max", c.max(), "08_transitions.csv", "max over learners")
ii = pd.read_csv(TAB / "08_errors_by_intensity.csv").query(
    "task == 'reg' and bin == '≥ 50 mm' and not model.str.startswith('baseline')", engine="python")
add("heavy_ratio_min", ii.ratio_fc_obs.min(), "08_errors_by_intensity.csv", "min forecast/observed, >= 50 mm, learners")
add("heavy_ratio_max", ii.ratio_fc_obs.max(), "08_errors_by_intensity.csv", "max forecast/observed, >= 50 mm, learners")

# ablation verdicts
ab = pd.read_csv(TAB / "06_ablation_master.csv")
add("ablation_contrasts", len(ab), "06_ablation_master.csv", "rows")
for v in ["noise", "model-specific", "solid"]:
    add(f"ablation_{v}", (ab.verdict == v).sum(), "06_ablation_master.csv", "count")

# tuning: seed-0 selection optimism = best trial (seed 0) - mean CV score of the re-run seeds 1-4
bud = pd.read_csv(TAB / "06_tuning_budget.csv").set_index(["task", "model"])
dvt6 = pd.read_csv(TAB / "06_default_vs_tuned.csv").set_index(["task", "model"])
for m in ["xgboost", "lightgbm", "catboost"]:
    seeds14 = dvt6.loc[("cls", m), "default"] + dvt6.loc[("cls", m), "Δ seeds 1-4"]
    add(f"seed0_optimism_{m}", bud.loc[("cls", m), "best_value_seed0"] - seeds14, "06_tuning_budget.csv + 06_default_vs_tuned.csv",
        "best trial value - mean BSS of seeds 1-4")
ridge = json.loads((ROOT / "models" / "optuna" / "best_params_ridge__reg.json").read_text())
add("ridge_alpha_retuned", ridge["best_params"]["alpha"], "models/optuna/best_params_ridge__reg.json", "re-tuned range 1e-4..1e5")

# heavy-rain lag (Stage 8 evidence stored as tuples)
import ast
ev = pd.read_csv(TAB / "08_limitations_evidence.csv", index_col=0)["value"]
for model, short in [("catboost_tweedie", "tweedie"), ("hurdle_catboost_log_msess", "hurdle")]:
    same, nxt, obs = ast.literal_eval(ev[f"{model}: mean forecast on >=50 mm days (same day / next day) vs observed mean"])
    lag0, prev = ast.literal_eval(ev[f"{model}: corr(forecast, obs) lag 0 / forecast vs previous-day obs"])
    for q, v in [("heavy_fc_sameday", same), ("heavy_fc_nextday", nxt), ("corr_target_day", lag0), ("corr_previous_day", prev)]:
        add(f"{short}_{q}", v, "08_limitations_evidence.csv", "parsed tuple")
n_dev, n_params = ast.literal_eval(ev["development days / max deep parameters (tuned TFT-cls)"])
add("tft_cls_tuned_parameters", n_params, "08_limitations_evidence.csv", "parsed tuple")

derived = pd.DataFrame(rows)
derived.to_csv(TAB / "09_derived_numbers.csv", index=False)
derived.assign(value=derived.value.round(4))[["quantity", "value", "source"]]

,quantity,value,source
0,test_days,2192.0000,07_test_by_season_cls.csv
1,test_wet_share,0.5566,08_errors_by_year.csv
2,test_mean_mm,14.1279,08_errors_by_year.csv
3,test_max_mm,202.3500,08_errors_by_year.csv
4,prectotcorr_mean_1981_1997,1924.8824,00_annual_precip_totals.csv
5,prectotcorr_mean_1998_2015,2970.2778,00_annual_precip_totals.csv
6,prectotcorr_mean_2016_2025,5377.2000,00_annual_precip_totals.csv
7,imerg_mean_1998_2015,3680.1667,00_annual_precip_totals.csv
8,imerg_mean_2016_2025,3757.1000,00_annual_precip_totals.csv
9,n_rows,16406.0000,data/processed/features_B.parquet


**Interpretation.** All derived numbers are plausible and agree with Stages 7–8: 2,192 test days, wet share 0.557, mean 14.13 mm/day, maximum 202.35 mm. Three values differ from earlier *write-ups*, and the CSVs win: (i) the ablation master table has **118** contrasts (68 noise / 20 model-specific / 30 solid), not the "84 (49/10/25)" in DECISIONS 6.13 and the Stage-8 limitation text. The CSV has had 119 lines since its Stage-6 commit, so the earlier count was a transcription error. (ii) The dev→test Spearman for BSS is **0.89** from the CSV, while the Stage-7 notebook printed 0.88 from unrounded scores; rounding to 3 decimals creates ties (e.g. LightGBM and CatBoost both 0.265). (iii) The 2016–2025 PRECTOTCORR mean is 5,377 mm, which DATA_SOURCE.md rounds to 5,380. The re-tuned Ridge α (236) is read from its Optuna JSON because `06_best_params.csv` still shows the first, boundary optimum (99.4).

**Decision.** The manuscript cites `09_derived_numbers.csv` for all ranges and counts; the three corrections are logged in DECISIONS 9.3 and in the issue list.

**Paper note.** Ranges and counts quoted in the text were computed reproducibly from the archived result tables rather than transcribed by hand.

**WHAT.** Measure how often the observed test-day rainfall fell inside TFT's central 80 % and 90 % prediction intervals, per seed.

**WHY.** Prompt 9.2 asks for TFT intervals on the test years, but Stage 7 did not store their coverage. This is a descriptive analysis of frozen predictions (like Stage 8): no model, quantile or threshold is changed.

**Alternatives considered.** Seed-averaged (Vincentised) quantiles — would describe a forecast nobody issued; per-seed coverage with mean ± SD is closer to what each model produced. Pinball loss — already reported for development; coverage is easier to read.

**What to look for.** Coverage close to the nominal 0.80 / 0.90. Values far below nominal would mean over-confident intervals; far above, intervals too wide.

In [3]:
import final_report as FR    # reads the frozen test predictions and labels (allowed only because the manifest exists)

_, _, extra = FR.load_predictions("reg")
te, _, y = FR.truth()
rows = []
for seed, q in extra["tft_quantiles"].items():
    qs = np.sort(q.loc[te, ["q0.05", "q0.1", "q0.25", "q0.5", "q0.75", "q0.9", "q0.95"]].to_numpy(), axis=1)
    obs = y.loc[te].to_numpy()
    rows.append({"seed": seed,
                 "coverage80": np.mean((obs >= qs[:, 1]) & (obs <= qs[:, 5])),
                 "coverage90": np.mean((obs >= qs[:, 0]) & (obs <= qs[:, 6])),
                 "share_above_q95": np.mean(obs > qs[:, 6]),
                 "mean_width80_mm": np.mean(qs[:, 5] - qs[:, 1])})
cov = pd.DataFrame(rows)
summary = cov.drop(columns="seed").agg(["mean", "std"]).T.reset_index(names="quantity")
summary.to_csv(TAB / "09_tft_interval_coverage_test.csv", index=False)
cov.round(3), summary.round(3)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(   seed  coverage80  coverage90  share_above_q95  mean_width80_mm
 0     0       0.861       0.919            0.067        22.055000
 1     1       0.857       0.927            0.047        24.805000
 2     2       0.867       0.934            0.053        24.615999
 3     3       0.844       0.917            0.072        21.261000
 4     4       0.834       0.927            0.067        21.983999,
           quantity    mean    std
 0       coverage80   0.852  0.013
 1       coverage90   0.925  0.007
 2  share_above_q95   0.061  0.011
 3  mean_width80_mm  22.944  1.644)

**Interpretation.** On the test years the TFT intervals covered **85.2 %** (80 % interval) and **92.5 %** (90 % interval) of days, with a seed SD of about 1 percentage point. 6.1 % of days lay above the 0.95 quantile. The intervals are therefore slightly *conservative* (too wide), unlike in 2015–2019 (82.1 % / 89.2 %). Part of this is the point mass at 0 mm: on dry days a lower quantile of 0 mm counts as covered, which inflates coverage.

**Decision.** Report the coverage as descriptive (Section 4.5) and disclose that it was computed after the one-shot run (Section 3.8). No model is changed.

**Paper note.** On the test years, the central 80 % and 90 % TFT prediction intervals covered 85.2 % and 92.5 % of days, indicating slightly conservative intervals for a zero-inflated target.

**WHAT.** Build the four publication tables from the Stage-7 CSVs: Table 1 (classification scores), Table 2 (regression scores), Table 3 and Table 4 (paired comparisons with Holm-adjusted DM p-values). Save each as CSV and LaTeX.

**WHY.** Tables are generated, not typed, so they cannot disagree with the evidence. Point estimates are the scores of the seed-mean forecast (the tested forecast); the seed SD shows seed-to-seed variability separately.

**Alternatives considered.** Copying the Stage-7 `07_table1/2` strings (they mix per-seed means with bootstrap CIs of the seed-mean forecast, which is confusing in a paper); showing all 13 metrics (unreadable).

**What to look for.** CatBoost BSS 0.376, CatBoost Tweedie MAE 8.534, and significance stars that match `07_significance_matrix.csv`.

In [4]:
NAMES = {"baseline:majority": "Majority / mean", "baseline:persistence": "Persistence",
         "baseline:climatology": "Climatology", "baseline:seasonal": "Seasonal",
         "baseline:seasonal+persistence": "Seasonal + persistence",
         "logreg_none": "Logistic regression", "logreg_l2": "Logistic L2 (ridge)", "logreg_l1": "Logistic L1 (lasso)",
         "logreg_enet": "Logistic elastic net", "sgd_logloss": "SGD (log loss)", "linreg": "Linear regression",
         "ridge": "Ridge", "lasso": "Lasso", "elasticnet": "Elastic net", "sgd_reg": "SGD regressor",
         "random_forest": "Random forest", "xgboost": "XGBoost", "lightgbm": "LightGBM", "catboost": "CatBoost",
         "mlp": "MLP", "mlp_huber": "MLP (Huber)", "catboost_tweedie": "CatBoost Tweedie",
         "hurdle_catboost_log_msess": "Hurdle CatBoost (log + smearing)",
         "deep:tsmixer": "TSMixer", "deep:tide": "TiDE", "deep:tft": "TFT"}
ci = lambda r, k, d=3: f"{r[k]:.{d}f} [{r[k + '_lo']:.{d}f}, {r[k + '_hi']:.{d}f}]"
pfmt = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"

def metric_table(task, cols_ci, cols_plain, sd_metric):
    boot = pd.read_csv(TAB / f"07_test_{task}_bootstrap.csv", index_col=0)
    seed = pd.read_csv(TAB / f"07_test_{task}_per_seed.csv")
    sd = seed.groupby("model")[sd_metric].std().fillna(0.0)
    n = seed.groupby("model").seed.nunique()
    fam = pd.read_csv(TAB / f"07_table{1 if task == 'cls' else 2}_{'classification' if task == 'cls' else 'regression'}.csv",
                      index_col=0)["family"]
    out = []
    for m, r in boot.iterrows():
        row = {"Model": NAMES[m], "Family": fam[m], "Seeds": int(n[m])}
        for k in cols_ci:
            row[f"{k} [95% CI]"] = ci(r, k)
            if k == sd_metric:
                row[f"{sd_metric} seed SD"] = f"{sd[m]:.3f}"
        for k in cols_plain:
            row[k] = "--" if pd.isna(r[k]) else f"{r[k]:.3f}"
        out.append(row)
    return pd.DataFrame(out)

t1 = metric_table("cls", ["BSS"], ["BSS_vs_seas_pers", "ROC_AUC", "PR_AUC", "CSI", "HSS", "ECE"], "BSS")
t2 = metric_table("reg", ["MAE", "MSESS"], ["MSESS_vs_seas_pers", "RMSE", "bias", "recall_ge50", "precision_ge50"], "MAE")

def test_table(task, specs):
    rows = {}
    for label, fam, loss in specs:
        sub = cmp.query("family == @fam and task == @task and loss == @loss")
        for r in sub.itertuples():
            star = "*" if r.significant else ""
            rows.setdefault(r.model, {"Model": NAMES[r.model]})
            rows[r.model][label] = f"{r.estimate:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]"
            rows[r.model][f"p_Holm ({label.split()[0]} {label.split()[-1]})"] = pfmt(r.p_holm) + star
    order = [m for m in NAMES if m in rows]
    return pd.DataFrame([rows[m] for m in order]).fillna("ref.")

t3 = test_table("cls", [("ΔBSS vs s+p", "A", "Brier"), ("ΔBSS vs LightGBM", "B", "Brier")])
t4 = test_table("reg", [("ΔMAE vs s+p", "A", "abs_error"), ("ΔMSESS vs s+p", "A", "sq_error"),
                        ("ΔMAE vs Tweedie", "B", "abs_error")])
PAPER_TABLES = {"TABLE 1": t1, "TABLE 2": t2, "TABLE 3": t3, "TABLE 4": t4}
for i, (name, t) in enumerate(PAPER_TABLES.items(), 1):
    t.to_csv(TAB / f"09_paper_table{i}.csv", index=False)
    (TAB / f"09_paper_table{i}.tex").write_text(t.to_latex(index=False, escape=True))
t1

,Model,Family,Seeds,BSS [95% CI],BSS seed SD,BSS_vs_seas_pers,ROC_AUC,PR_AUC,CSI,HSS,ECE
0,Majority / mean,baseline,1,"-1.529 [-2.134, -1.106]",0.000,-2.457,0.500,0.557,0.000,0.000,0.062
1,Persistence,baseline,1,"0.130 [0.011, 0.228]",0.000,-0.189,0.908,0.888,0.848,0.815,0.057
2,Climatology,baseline,1,"0.000 [0.000, 0.000]",0.000,-0.367,0.937,0.945,0.808,0.764,0.063
3,Seasonal,baseline,1,"-0.011 [-0.023, 0.002]",0.000,-0.383,0.937,0.947,0.807,0.764,0.065
4,Seasonal + persistence,baseline,1,"0.269 [0.213, 0.321]",0.000,0.000,0.960,0.964,0.858,0.826,0.053
5,Logistic regression,classical,1,"0.360 [0.293, 0.416]",0.000,0.124,0.969,0.975,0.862,0.827,0.018
6,Logistic L2 (ridge),classical,1,"0.358 [0.300, 0.408]",0.000,0.123,0.970,0.975,0.867,0.836,0.030
7,Logistic L1 (lasso),classical,5,"0.351 [0.293, 0.402]",0.000,0.112,0.970,0.976,0.867,0.838,0.037
8,Logistic elastic net,classical,5,"0.358 [0.299, 0.408]",0.000,0.122,0.970,0.975,0.868,0.837,0.029
9,SGD (log loss),classical,5,"0.360 [0.299, 0.409]",0.000,0.124,0.970,0.975,0.866,0.833,0.028


**Interpretation.** The tables reproduce the Stage-7 numbers: CatBoost BSS 0.376 [0.324, 0.423] and CatBoost Tweedie MAE 8.534 [6.711, 10.359]. The stars in Tables 3–4 match `07_significance_matrix.csv`. The seed SD is small for the classical models (≤ 0.006 BSS) and largest for TFT (0.018 BSS) and TSMixer regression (1.919 mm MAE, the failed seed 1). This is why the *seed-mean* forecast is the tested forecast.

**Decision.** Use these four generated tables in the paper; Table 1 and Table 2 report the seed-mean forecast with bootstrap CIs plus the per-seed SD.

**Paper note.** All tables were generated programmatically from the frozen test predictions; point estimates refer to the seed-mean forecast and are accompanied by 95 % monthly block-bootstrap intervals and the across-seed standard deviation.

**WHAT.** Insert the generated tables into the manuscript between the `<!-- BEGIN TABLE n -->` / `<!-- END TABLE n -->` markers.

**WHY.** The prose and the tables then live in one document that is always regenerated from the CSVs.

**Alternatives considered.** Keeping the tables in separate files only (reviewers read one document).

**What to look for.** Four tables inserted; re-running is idempotent (the markers are kept).

In [5]:
md = DRAFT.read_text()
for name, t in PAPER_TABLES.items():
    md = P.insert_block(md, name, P.markdown_table(t))
DRAFT.write_text(md)
print("tables inserted:", [n for n in PAPER_TABLES], "| draft lines:", md.count("\n"))

tables inserted: ['TABLE 1', 'TABLE 2', 'TABLE 3', 'TABLE 4'] | draft lines: 320


**Interpretation.** The four tables are now inside `paper_draft.md`; the markers stay, so re-running replaces the tables instead of duplicating them.

**Decision.** The Markdown file is the single source of the manuscript; the LaTeX file is generated from it at the end.

**Paper note.** (Workflow step; no paper sentence.)

**WHAT.** Define the hostile-review checks: (1) every stated number equals its CSV value; (2) it appears in its sentence; (3) no untraced number in the Abstract or Results; (4) every significance claim has a Holm source or says it is uncorrected; (5) abbreviations defined at first use; (6) consistent terminology; (7) citations ⊆ the 17 verified bib entries and vice versa; (8) all baselines and model families described in Methods; (9) all ten Stage-8 limitations present.

**WHY.** Prompt 9.4 asks for a *programmatic* cross-check, so the review is repeatable and not limited by my attention.

**Alternatives considered.** Manual proof-reading only (misses numbers); regex-extracting every number and searching all CSVs for it (many false matches, e.g. 0.37 appears in dozens of tables).

**What to look for.** Nothing printed yet: this cell only defines the checks. The constants in `ALLOWED` are protocol settings defined in Methods (thresholds, CI level, seeds), not results.

In [6]:
# names explained in the variables table or proper names, not abbreviations needing a definition
KNOWN = {"PRECTOTCORR", "IMERG_PRECTOT", "T2M", "T2M_MAX", "T2M_MIN", "T2MDEW", "RH2M", "QV2M", "PS", "WS2M", "WS10M",
         "WD2M", "ALLSKY_SFC_SW_DWN", "YEAR", "DOY", "XGBoost", "CatBoost", "TSMixer", "SHA-256", "SD", "SYN1deg",
         "L1", "L2", "URL", "CSV", "UTC+6", "SHapley"}
# protocol constants defined in Section 3 (thresholds, CI level, resamples, seeds, look-backs, feature counts)
ALLOWED = {"0.1", "2.5", "20", "50", "80", "90", "95", "1,000", "42", "14", "30", "60", "0.95", "71", "115", "0.001"}
VARIANTS = {"seasonal + persistence": ["seasonal + persistence", "seasonal+persistence", "seasonal-plus-persistence",
                                       "seasonal and persistence baseline"],
            "Brier skill score": ["Brier skill score", "Brier Skill Score"],
            "hurdle model": ["hurdle model", "two-part model"],
            "CatBoost Tweedie": ["CatBoost Tweedie", "Tweedie CatBoost", "catboost_tweedie"],
            "L1-logistic": ["L1-logistic", "lasso-logistic", "logreg_l1"],
            "development": ["development years", "training years"]}
BASELINES = ["majority", "persistence", "climatology", "seasonal model", "seasonal + persistence"]
MODELS = ["logistic regression", "ridge", "lasso", "elastic-net", "SGD", "random forest", "XGBoost", "LightGBM",
          "CatBoost", "multilayer perceptron", "TSMixer", "TiDE", "TFT", "linear"]
LIMITS = {"reanalysis target": "Reanalysis target", "single cell": "Single grid cell", "inhomogeneity": "Inhomogeneous",
          "seasonal imbalance": "Seasonal imbalance", "heavy rain": "Heavy rain", "sample size": "Sample size",
          "hindcast / NWP / UTC / latency": "latency", "non-stationarity": "Non-stationarity",
          "single test period": "Single test period", "interpretability": "Interpretability"}

def section(md, title):
    m = re.search(rf"^# {re.escape(title)}.*?(?=^# )", md, re.S | re.M)
    return m.group() if m else ""

def review(md):
    """All hostile-review checks on one manuscript text -> {check: DataFrame of problems}."""
    src = P.check_sources(md)
    bib = set(P.bib_keys())
    cites = set(P.citation_keys(md))
    methods = section(md, "3 Methods").lower()
    lim = section(md, "6 Limitations")
    out = {
        "number does not match CSV": src[~src.value_ok],
        "number not in the sentence": src[src.value_ok & ~src.in_text],
        "untraced number (Abstract, Results)": P.untraced_numbers(md, allowed=ALLOWED),
        "significance claim without Holm/hedge": P.significance_claims(md).query("not ok"),
        "abbreviation not defined at first use": P.abbreviations(md, KNOWN),
        "inconsistent terminology": P.terminology(md, VARIANTS).query("not consistent"),
        "citation key missing from bib": pd.DataFrame({"key": sorted(cites - bib)}),
        "bib entry never cited": pd.DataFrame({"key": sorted(bib - cites)}),
        "baseline not described in Methods": pd.DataFrame({"baseline": [b for b in BASELINES if b not in methods]}),
        "model not described in Methods": pd.DataFrame({"model": [m for m in MODELS if m.lower() not in methods]}),
        "limitation missing": pd.DataFrame({"limitation": [k for k, v in LIMITS.items() if v not in lim]}),
    }
    return src, out

def summary(out):
    return pd.Series({k: len(v) for k, v in out.items()}, name="problems")

**Interpretation.** The checks are defined. The whitelist `ALLOWED` contains only protocol constants defined in Methods (thresholds 0.1/2.5 mm, heavy-rain classes 20/50 mm, interval levels 80/90/95 %, 1,000 resamples, seed 42, look-backs 14/30/60, feature counts 71/115, the reporting bound 0.001). `KNOWN` contains variable names explained in the variables table and proper names, not undefined jargon.

**Decision.** Apply the identical checks to the first and the final draft.

**Paper note.** Manuscript consistency was verified automatically: every reported number was linked to its source table and checked for correct rounding, and every significance statement was linked to a Holm-adjusted test.

**WHAT.** Run the review on the **first draft** (`paper_draft_v0.md`, the snapshot taken before any review).

**WHY.** Shows honestly which problems the first draft had, so the fixes are documented rather than invisible.

**Alternatives considered.** Reviewing only the final draft (would hide what was fixed).

**What to look for.** Ideally zero numerical mismatches; text-level problems (missing definitions, unhedged claims) are expected in a first draft.

In [7]:
src0, out0 = review(DRAFT_V0.read_text())
print(f"first draft: {len(src0)} stated numbers checked against the CSVs")
summary(out0)

first draft: 277 stated numbers checked against the CSVs


number does not match CSV                0
number not in the sentence               8
untraced number (Abstract, Results)      0
significance claim without Holm/hedge    3
abbreviation not defined at first use    6
inconsistent terminology                 0
citation key missing from bib            0
bib entry never cited                    0
baseline not described in Methods        0
model not described in Methods           0
limitation missing                       0
Name: problems, dtype: int64

**Interpretation.** The first draft had **zero numerical mismatches** in 277 stated numbers. The checker itself needed three fixes before this run (sign handling of negative numbers, non-source HTML comments, hyphenated acronyms such as MERRA-2), and those are documented here. The draft did have 17 text-level problems: 8 numbers named in source comments but not written in the sentence, 3 significance statements in the Discussion and Conclusion without a Holm reference, and 6 abbreviations used before definition (NASA, GPM, CERES, SHAP, ROC, and IMERG in the Abstract).

**Decision.** Fix each one in the text (next cells list them with line numbers).

**Paper note.** (Review step; no paper sentence.)

**WHAT.** List the details of every problem found in the first draft.

**WHY.** Each problem needs a location and a fix in the issue list.

**Alternatives considered.** Printing counts only.

**What to look for.** Line numbers in v0, the stated number or abbreviation, and the check that failed.

In [8]:
for k, v in out0.items():
    if len(v):
        print(f"\n== {k} ({len(v)})")
        print(v.drop(columns=[c for c in ["section", "first_use", "excerpt", "csv_value", "key"] if c in v]).to_string(
            max_colwidth=60))


== number not in the sentence (8)
     line                               file                           name stated  value_ok  in_text
8      14             09_derived_numbers.csv    famB_cls_learner_p_holm_min  0.065      True    False
95    123             09_derived_numbers.csv  famA_cls_learners_significant     12      True    False
132   179      07_statistical_comparison.csv                         p_holm  0.053      True    False
146   181      07_statistical_comparison.csv                         p_holm  0.030      True    False
148   181      07_statistical_comparison.csv                         p_holm  0.020      True    False
185   245      07_statistical_comparison.csv                         p_holm  0.000      True    False
187   245      07_statistical_comparison.csv                         p_holm  0.002      True    False
190   245  09_tft_interval_coverage_test.csv                            std   1.3%      True    False

== significance claim without Holm/hedge (3)
 

**Interpretation.** The details show *what kind* of errors a first draft makes. They are not wrong numbers but **unverifiable statements**: a p-value cited in a comment but missing from the sentence, so a reader could not see it, and an abbreviation used before it was defined. The three unhedged significance claims summarised Holm results correctly, but without a pointer to the test.

**Decision.** Add the p-values to the sentences, define the abbreviations at first use, and attach Holm sources to the Discussion and Conclusion claims.

**Paper note.** (Review step; no paper sentence.)

**WHAT.** Assemble the numbered issue list: the programmatic findings on v0 plus the findings of a manual hostile-reviewer reading (over-claims, missing baselines, leakage risks, inconsistent records, missing references). Save as `09_review_issues.csv`.

**WHY.** Prompt 9.4: a numbered issue list with locations and the fix applied. Open items that only the researcher can resolve are kept open, not hidden.

**Alternatives considered.** Fixing silently (violates rule 12).

**What to look for.** All numerical and claim issues fixed; only bibliographic / authorship items remain open.

In [9]:
auto = []
for check, df in out0.items():
    for r in df.to_dict("records"):
        what = r.get("name") or r.get("abbreviation") or r.get("concept") or r.get("excerpt", "")[:50]
        auto.append({"location": f"v0 line {r.get('line', '-')}", "check": check, "problem": str(what),
                     "fix": "text completed / definition added / Holm p-value added", "status": "fixed"})
manual = [  # hostile-reviewer reading of the whole draft (location in the final draft)
    ("Title, Sec. 1, Sec. 7", "'pre-registered' overstates an internal freeze (no public registry)", "reworded: frozen protocol / internal pre-registration", "fixed"),
    ("Abstract", "'none differed significantly from LightGBM' is false for TiDE", "'none of these twelve' + smallest p_Holm", "fixed"),
    ("Sec. 4.5, Sec. 5", "non-significance read as equivalence ('indistinguishable', 'matched')", "'not significantly different'", "fixed"),
    ("Sec. 3.4, 3.5", "internal stage jargon ('Stage-1', 'Stage-5 default')", "plain descriptions", "fixed"),
    ("Sec. 3.8, 4.5", "TFT test coverage computed after the one-shot run (post-hoc test analysis)", "disclosed as descriptive; no model or test changed", "fixed"),
    ("Sec. 6 (7)", "no comparison with operational NWP / BMD forecasts (missing baseline)", "stated as limitation", "fixed"),
    ("Sec. 4.6, 6 (9)", "DECISIONS 6.13 / Stage-8 limitation text say 84 ablation contrasts (49/10/25); 06_ablation_master.csv has 118 (68/20/30) since Stage 6", "paper uses the CSV; correction logged (DECISIONS 9.3)", "fixed"),
    ("Sec. 4.3", "dev->test Spearman for BSS: 0.88 in the Stage-7 notebook (unrounded scores) vs 0.89 from the rounded CSV (ties)", "paper reports 0.89 from the CSV; logged", "fixed"),
    ("Sec. 4.1", "DATA_SOURCE.md rounds the 2016-2025 mean to 5,380 mm; CSV gives 5,377 mm", "paper uses 5,377", "fixed"),
    ("Sec. 4.2", "06_best_params.csv still shows the first Ridge optimum (alpha 99.4, upper edge); the re-tuned alpha 236 is only in the JSON", "derived table reads the JSON", "fixed"),
    ("Sec. 8", "unit-test count 107 became 114 with the checker tests", "count stated with breakdown; verified by pytest below", "fixed"),
    ("requirements.txt", "nbformat, nbconvert, ipykernel, nbclient were unpinned", "pinned to the installed versions", "fixed"),
    ("Sec. 3.7, 3.5, 3.8, 4.1", "methods without a verified reference (Holm, HAC, block bootstrap, Pettitt, Optuna, SHAP, ALE, Tweedie, PyTorch)", "marked [NEEDS CITATION]", "open: researcher"),
    ("references.bib", "Ahmed et al. (2021): title and author list not in the supplied list", "marked [CHECK]", "open: researcher"),
    ("Front matter, Sec. 2.2, 8", "authors, repository URL, data citation, IMERG header label", "marked [CHECK]", "open: researcher"),
]
issues = pd.DataFrame(auto + [dict(zip(["location", "problem", "fix", "status"], m), check="manual review") for m in manual])
issues.index = range(1, len(issues) + 1)
issues.index.name = "issue"
issues.to_csv(TAB / "09_review_issues.csv")
print(issues.status.value_counts().to_string())
issues[["location", "check", "problem", "status"]]

status
fixed               29
open: researcher     3


,location,check,problem,status
issue,,,,
1,v0 line 14,number not in the sentence,famB_cls_learner_p_holm_min,fixed
2,v0 line 123,number not in the sentence,famA_cls_learners_significant,fixed
3,v0 line 179,number not in the sentence,p_holm,fixed
4,v0 line 181,number not in the sentence,p_holm,fixed
5,v0 line 181,number not in the sentence,p_holm,fixed
6,v0 line 245,number not in the sentence,p_holm,fixed
7,v0 line 245,number not in the sentence,p_holm,fixed
8,v0 line 245,number not in the sentence,std,fixed
9,v0 line 265,significance claim without Holm/hedge,The central result is that machine learning added,fixed


**Interpretation.** **32 issues**: 17 found programmatically on v0 and 15 by the manual hostile reading. 29 are fixed. The most important manual findings were:
- "pre-registered" over-stated an internal freeze;
- the Abstract's "none differed from LightGBM" was false for TiDE;
- non-significance had been read as equivalence ("statistically indistinguishable");
- the post-hoc TFT coverage on the test years needed disclosure;
- the missing comparison with operational NWP/BMD forecasts had to be stated as a limitation;
- inconsistent records (84 vs 118 ablation contrasts, Spearman 0.88 vs 0.89, 5,380 vs 5,377 mm, Ridge α);
- the test count and unpinned packages needed correcting.

Three issues stay **open for the researcher**: references outside the verified list, the Ahmed et al. (2021) title, and author/URL/data-citation details.

**Decision.** Keep the open items visibly marked (`[CHECK]`, `[NEEDS CITATION]`) instead of guessing references (rule 13).

**Paper note.** (Review record; see `reports/tables/09_review_issues.csv`.)

**WHAT.** Run the identical review on the **final** draft and stop with an error if any problem remains; save the full number-by-number cross-check.

**WHY.** The deliverable requires zero unresolved numerical discrepancies; an `assert` makes that a hard condition of the notebook.

**Alternatives considered.** Reporting a percentage of verified numbers (would tolerate errors).

**What to look for.** All eleven counts are 0, and 100 % of stated numbers verified.

In [10]:
md = DRAFT.read_text()
src1, out1 = review(md)
counts = summary(out1)
print(f"final draft: {len(src1)} stated numbers, {src1.file.nunique()} source files, "
      f"{(src1.value_ok & src1.in_text).mean():.0%} verified")
assert counts.sum() == 0, counts[counts > 0]
src1.drop(columns=["section"]).to_csv(TAB / "09_number_crosscheck.csv", index=False)
counts

final draft: 282 stated numbers, 43 source files, 100% verified


number does not match CSV                0
number not in the sentence               0
untraced number (Abstract, Results)      0
significance claim without Holm/hedge    0
abbreviation not defined at first use    0
inconsistent terminology                 0
citation key missing from bib            0
bib entry never cited                    0
baseline not described in Methods        0
model not described in Methods           0
limitation missing                       0
Name: problems, dtype: int64

**Interpretation.** The final draft passes **all eleven checks with zero problems**: 282 stated numbers from 43 source files, 100 % verified for value *and* presence in the sentence. The `assert` guarantees that this notebook cannot finish if a discrepancy is introduced later.

**Decision.** The manuscript has zero unresolved numerical discrepancies; `09_number_crosscheck.csv` is the audit trail.

**Paper note.** All 282 source-linked numbers in the manuscript were verified automatically against the archived result tables.

**WHAT.** Independently recompute three headline numbers directly from the frozen test predictions: CatBoost BSS, CatBoost Tweedie MAE and the raw DM p-value of Tweedie vs seasonal + persistence.

**WHY.** The checker only proves that the paper equals the CSVs. A hostile reviewer would also ask whether the CSVs equal the predictions. Rule 11: a high score is checked, not trusted.

**Alternatives considered.** Re-running the full bootstrap (slow, same code path as Stage 7); trusting the tables.

**What to look for.** Exact agreement (differences at rounding level 1e-12).

In [11]:
import final_eval as FE
import metrics as M
# recompute three headline numbers from the frozen predictions themselves (not from the Stage-7 tables)
pc, tc, _ = FR.load_predictions("cls"); pr, _, _ = FR.load_predictions("reg")
te, ycls, yreg = FR.truth()
p = {m: P_.mean(axis=1).to_numpy() for m, P_ in pc.items()}
a = {m: P_.mean(axis=1).to_numpy() for m, P_ in pr.items()}
brier = lambda q: np.mean((q - ycls.to_numpy()) ** 2)
spot = pd.DataFrame([
    ("CatBoost BSS", 1 - brier(p["catboost"]) / brier(p["baseline:climatology"]),
     P.lookup("07_test_cls_bootstrap.csv", "index=catboost", "BSS")),
    ("CatBoost Tweedie MAE", np.mean(np.abs(a["catboost_tweedie"] - yreg.to_numpy())),
     P.lookup("07_test_reg_bootstrap.csv", "index=catboost_tweedie", "MAE")),
    ("Tweedie vs s+p DM raw p", FE.dm_test(np.abs(a["catboost_tweedie"] - yreg.to_numpy()),
                                             np.abs(a["baseline:seasonal+persistence"] - yreg.to_numpy()))[2],
     P.lookup("07_statistical_comparison.csv", "family=A & task=reg & loss=abs_error & model=catboost_tweedie", "p")),
], columns=["quantity", "recomputed", "table"])
spot["agree"] = np.isclose(spot.recomputed, spot.table, rtol=1e-9, atol=1e-12)
assert spot.agree.all()
spot

,quantity,recomputed,table,agree
0,CatBoost BSS,0.376308,0.376308,True
1,CatBoost Tweedie MAE,8.533666,8.533666,True
2,Tweedie vs s+p DM raw p,0.022827,0.022827,True


**Interpretation.** The three headline numbers recomputed directly from the frozen predictions agree with the Stage-7 tables to machine precision: CatBoost BSS 0.376308, Tweedie MAE 8.533666 mm and DM p = 0.022827. So the chain predictions → CSV → manuscript is closed at both ends.

**Decision.** No correction needed; the Stage-7 tables are trustworthy.

**Paper note.** Headline test scores were independently recomputed from the archived predictions and matched the reported values exactly.

**WHAT.** Count the open review markers: `[CHECK]` (statements I could not trace in the repository) and `[NEEDS CITATION]` (claims that need a reference outside the 16 verified ones).

**WHY.** Prompt 9.3 forbids citing anything outside the verified list; the markers make the gaps visible to the researcher.

**Alternatives considered.** Filling the gaps from memory (risk of invented references, rule 13).

**What to look for.** A short list, mostly standard methods (Holm, HAC, bootstrap, Pettitt, Optuna, SHAP, ALE, Tweedie, PyTorch).

In [12]:
mk = P.markers(md)
print(mk.marker.value_counts().to_string())
mk.groupby(["marker", "section"]).size().rename("n").reset_index()

marker
[NEEDS CITATION]    15
[CHECK]              3


,marker,section,n
0,[CHECK],,1
1,[CHECK],2.2 NASA POWER daily data,1
2,[CHECK],2.3 Data audit and cleaning,1
3,[NEEDS CITATION],1 Introduction,3
4,[NEEDS CITATION],3.4 Models,2
5,[NEEDS CITATION],3.5 Hyper-parameter optimisation,1
6,[NEEDS CITATION],3.7 Statistical testing,3
7,[NEEDS CITATION],"3.8 Ablations, error analysis and interpretability",2
8,[NEEDS CITATION],4.1 Data audit and baselines,1
9,[NEEDS CITATION],5 Discussion,3


**Interpretation.** There are 15 `[NEEDS CITATION]` and 3 `[CHECK]` markers. The citations are standard methods or context claims not covered by the 16 verified references: flash floods in Sylhet, NWP practice, Holm, the HAC variance, the block bootstrap, Pettitt, Optuna/TPE, Tweedie loss, PyTorch, SHAP, ALE, BMD gauges and nor'westers. The `[CHECK]` items are the author list, the IMERG header label in the POWER file and the explanation of the PRECTOTCORR breaks. The bib entry of Ahmed et al. (2021) and the repository URL also carry `[CHECK]`.

**Decision.** Leave them for the researcher; I do not add references that are not in the verified list.

**Paper note.** (Editorial; no paper sentence.)

**WHAT.** Convert the checked Markdown to LaTeX with pandoc (natbib citations; each `<!-- source -->` comment becomes a `% source:` comment).

**WHY.** One source of truth: the `.tex` file is generated from the verified `.md`, so the two cannot drift apart.

**Alternatives considered.** Maintaining the LaTeX by hand (two documents to keep consistent); pandoc's citeproc (would format references itself and lose the BibTeX file).

**What to look for.** Same number of `% source:` lines as source comments, 17 citation keys, 5 tables (4 results + variables), and a `\bibliography` command.

In [13]:
tex_path = P.to_latex(DRAFT, ROOT / "reports" / "paper_draft.tex")
tex = tex_path.read_text()
n_comments = len(P.COMMENT.findall(md))
checks = pd.Series({
    "source comments in Markdown": n_comments,
    "% source: lines in LaTeX": tex.count("% source:"),
    "citation keys in LaTeX": len({k.strip() for g in re.findall(r"\\cite[pt]?\{([^}]*)\}", tex) for k in g.split(",")}),
    "tables (longtable) in LaTeX": tex.count("\\begin{longtable}"),
    "\\bibliography present": int("\\bibliography{references}" in tex),
})
print("Unicode symbols (Δ, ≤, ρ, κ) are kept: compile with XeLaTeX or LuaLaTeX, then BibTeX.")
assert checks["% source: lines in LaTeX"] == n_comments and checks["\\bibliography present"] == 1
checks

Unicode symbols (Δ, ≤, ρ, κ) are kept: compile with XeLaTeX or LuaLaTeX, then BibTeX.


source comments in Markdown    163
% source: lines in LaTeX       163
citation keys in LaTeX          17
tables (longtable) in LaTeX      5
\bibliography present            1
dtype: int64

**Interpretation.** The LaTeX file has the same 163 source comments (now `% source:`), all 17 citation keys as natbib `\citep`/`\citet`, the 5 tables (4 results tables plus the variables table) and a `\bibliography{references}` command. The Unicode symbols (Δ, ≤, ρ, κ) need XeLaTeX or LuaLaTeX; pdfLaTeX would need extra packages. No LaTeX engine is installed in this environment, so the `.tex` file was not compiled here.

**Decision.** Ship `paper_draft.tex` as generated; compiling it is left to the researcher's TeX installation (an honest limitation of this check).

**Paper note.** (Workflow; no paper sentence.)

**WHAT.** Complete the **Pitfalls Checklist**, with machine-checked evidence where possible (frozen files unchanged since the freeze commit, test predictions unchanged since the Stage-7 commit, manifest complete).

**WHY.** Prompt 9.4; these are the classic ways forecasting papers go wrong.

**Alternatives considered.** A free-text checklist without evidence.

**What to look for.** Every pitfall 'avoided' or 'addressed', and the reanalysis self-consistency honestly listed as a limitation.

In [14]:
git = lambda *a: subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True)
frozen_ok = git("diff", "--quiet", "55d416f", "HEAD", "--", "final_config.yaml", "src/final_eval.py").returncode == 0
test_ok = git("diff", "--quiet", "4f8fd7b", "HEAD", "--", "models/test").returncode == 0
manifest = json.loads((ROOT / "models" / "test" / "manifest.json").read_text())
n_pred = len(list((ROOT / "models" / "test").glob("*.parquet")))
pitfalls = pd.DataFrame([
    ("Random / shuffled splits", "avoided", "tests/test_splits.py; expanding folds by target date, 1-day purge"),
    ("Future information in features", "avoided", "tests/test_leakage.py (truncation test with leaky control); 02_leakage_*.csv"),
    ("Preprocessing fitted on validation/test rows", "avoided", "pipelines; tests/test_experiment.py spy test; tests/test_tuning.py"),
    ("Tuning, thresholds or early stopping on the validation/test block", "avoided", "inner 3-year slice; tests/test_tuning.py label scrambling"),
    ("Repeated looks at the test set", "avoided", f"one run, manifest {manifest['commit'][:7]}, {n_pred} prediction files; models/test unchanged: {test_ok}"),
    ("Post-hoc change of model, threshold or test", "avoided", f"final_config.yaml and src/final_eval.py unchanged since freeze: {frozen_ok}"),
    ("Weak or missing baselines", "addressed", "5 baselines incl. seasonal + persistence; no NWP/BMD baseline (Limitation 7)"),
    ("Single seed / seed cherry-picking", "addressed", "5 seeds, seed-mean forecasts tested; seeds 1-4 optimism check"),
    ("Significance without multiplicity control", "addressed", "DM + HLN + Holm for test claims; ablations labelled uncorrected"),
    ("Inflated skill from a degraded reference", "addressed", "BSS vs seasonal + persistence reported; 07_dev_vs_test_ranking.csv"),
    ("Accuracy on a skewed target", "addressed", "BSS, CSI, HSS, PR-AUC, MAE+MSESS+bias, heavy-rain recall"),
    ("Back-transformation bias of log targets", "addressed", "Duan smearing; 04_jensen_log_backtransform.csv"),
    ("Target/predictor self-consistency (reanalysis)", "limitation", "IMERG secondary check; Limitation 1"),
    ("Numbers in the paper not traceable", "addressed", f"{len(src1)} source-linked numbers, 0 discrepancies (09_number_crosscheck.csv)"),
], columns=["pitfall", "status", "evidence"])
assert frozen_ok and test_ok and len(manifest["files"]) == n_pred
pitfalls

,pitfall,status,evidence
0,Random / shuffled splits,avoided,"tests/test_splits.py; expanding folds by target date, 1-day purge"
1,Future information in features,avoided,tests/test_leakage.py (truncation test with leaky control); 02_leakage_*.csv
2,Preprocessing fitted on validation/test rows,avoided,pipelines; tests/test_experiment.py spy test; tests/test_tuning.py
3,"Tuning, thresholds or early stopping on the validation/test block",avoided,inner 3-year slice; tests/test_tuning.py label scrambling
4,Repeated looks at the test set,avoided,"one run, manifest 55d416f, 126 prediction files; models/test unchanged: True"
5,"Post-hoc change of model, threshold or test",avoided,final_config.yaml and src/final_eval.py unchanged since freeze: True
6,Weak or missing baselines,addressed,5 baselines incl. seasonal + persistence; no NWP/BMD baseline (Limitation 7)
7,Single seed / seed cherry-picking,addressed,"5 seeds, seed-mean forecasts tested; seeds 1-4 optimism check"
8,Significance without multiplicity control,addressed,DM + HLN + Holm for test claims; ablations labelled uncorrected
9,Inflated skill from a degraded reference,addressed,BSS vs seasonal + persistence reported; 07_dev_vs_test_ranking.csv


**Interpretation.** Every classic pitfall is either avoided with machine-checked evidence or addressed. `final_config.yaml` and `src/final_eval.py` are byte-identical to the freeze commit `55d416f`. `models/test/` (126 files plus the manifest) is unchanged since the Stage-7 commit `4f8fd7b`. One pitfall cannot be fixed by protocol: inputs and target come from the same reanalysis (self-consistency), so it is a stated limitation.

**Decision.** Include the checklist outcome in the Reproducibility section and the limitations.

**Paper note.** The evaluation followed a chronological, leakage-tested and frozen protocol in which the test period was used exactly once.

**WHAT.** Complete the **Reproducibility Checklist**: raw-data checksum, pinned packages, frozen config, manifest, number of unit tests (collected by pytest), notebooks, figures, number cross-check and LaTeX.

**WHY.** Prompt 9.3/9.4: the paper's Reproducibility section must be true, and this cell proves it.

**Alternatives considered.** Stating the items in prose only.

**What to look for.** All items `True`; 114 tests collected.

In [15]:
import hashlib
raw = ROOT / "data" / "raw" / "sylhet_power_daily_AG_1981-2025_raw.csv"
raw_sha = hashlib.sha256(raw.read_bytes()).hexdigest()
reqs = [l.strip() for l in (ROOT / "requirements.txt").read_text().splitlines() if l.strip()]
collected = subprocess.run([sys.executable, "-m", "pytest", "--collect-only", "-q", str(ROOT / "tests")], cwd=ROOT,
                           capture_output=True, text=True).stdout
n_tests = int(re.search(r"(\d+) tests? collected", collected).group(1))
notebooks = sorted(p.name for p in (ROOT / "notebooks").glob("*.ipynb"))
from PIL import Image
dpis = [Image.open(f).info.get("dpi", (0, 0))[0] for f in sorted((ROOT / "reports" / "figures").glob("*.png"))]
repro = pd.DataFrame([
    ("Raw data checksum matches DATA_SOURCE.md", raw_sha in (ROOT / "data" / "raw" / "DATA_SOURCE.md").read_text(), raw_sha[:16] + "..."),
    ("All packages pinned in requirements.txt", all("==" in r for r in reqs), f"{len(reqs)} packages"),
    ("Frozen configuration present", (ROOT / "final_config.yaml").exists(), "final_config.yaml (pre-test commit 827eae8)"),
    ("Test predictions + manifest present", (ROOT / "models" / "test" / "manifest.json").exists(), f"{n_pred} files, config sha {manifest['config_sha256'][:12]}..."),
    ("Unit tests collected", n_tests == 114, f"{n_tests} tests"),
    ("Seeds fixed", True, "models 0-4; Optuna TPE and bootstrap 42; numpy seeds in notebooks"),
    ("One notebook per step, outputs on disk", len(notebooks) >= 15, f"{len(notebooks)} notebooks"),
    ("Figures saved at 300 dpi", all(abs(d - 300) < 1 for d in dpis), f"{len(dpis)} PNG files in reports/figures"),
    ("Manuscript numbers machine-checked", counts.sum() == 0, "09_number_crosscheck.csv"),
    ("LaTeX generated from the checked Markdown", tex_path.exists(), "reports/paper_draft.tex (pandoc)"),
], columns=["item", "ok", "evidence"])
assert repro.ok.all(), repro[~repro.ok]
repro

,item,ok,evidence
0,Raw data checksum matches DATA_SOURCE.md,True,396f799b5cf6a551...
1,All packages pinned in requirements.txt,True,21 packages
2,Frozen configuration present,True,final_config.yaml (pre-test commit 827eae8)
3,Test predictions + manifest present,True,"126 files, config sha 205d4851bd72..."
4,Unit tests collected,True,114 tests
5,Seeds fixed,True,models 0-4; Optuna TPE and bootstrap 42; numpy seeds in notebooks
6,"One notebook per step, outputs on disk",True,15 notebooks
7,Figures saved at 300 dpi,True,61 PNG files in reports/figures
8,Manuscript numbers machine-checked,True,09_number_crosscheck.csv
9,LaTeX generated from the checked Markdown,True,reports/paper_draft.tex (pandoc)


**Interpretation.** Reproducibility is verified, not asserted. The raw-file SHA-256 matches DATA_SOURCE.md, and all 21 packages are pinned: four notebook packages were unpinned until this stage. The frozen config and manifest are present, pytest collects 114 tests (107 pipeline + 7 checker), and all 61 figures are 300 dpi. The manuscript numbers and the LaTeX file are generated and checked.

**Decision.** Stage 9 is complete; the remaining work is editorial (open markers).

**Paper note.** Code, data checksums, pinned software versions, the frozen configuration, the test-prediction manifest and 114 unit tests are provided to reproduce every reported number.

## END-OF-STAGE 9

**Five key things**
1. The manuscript (`reports/paper_draft.md` → `paper_draft.tex`, `references.bib`) is generated from the CSV tables. **282 numbers** carry source comments and are machine-checked for value and presence: 0 discrepancies.
2. The headline is modest but solid: every classical model, TFT and TSMixer beat seasonal + persistence in BSS (+0.070 to +0.108, Holm p ≤ 0.0001), with no significant differences among them. No model significantly improved MAE.
3. The biggest caveat is the target: against independent IMERG rain the frozen forecasts keep their discrimination (AUC ≈ 0.92) but lose their probabilistic skill (BSS ≈ 0).
4. The hostile review found 32 issues. The first draft had **no wrong numbers** but 17 unverifiable or under-defined statements and several over-claims ("pre-registered", "indistinguishable"). It also exposed inconsistent earlier records (84 vs 118 ablation contrasts; Spearman 0.88 vs 0.89).
5. Reproducibility is proved by checks rather than claims: data hash, pinned packages, frozen files unchanged since the freeze, 114 tests, 300-dpi figures, and predictions → tables → paper recomputed at both ends.

**Three questions to check understanding**
1. Why does the paper headline BSS *against seasonal + persistence* rather than against climatology, although climatology is the textbook reference? What happened to the climatology reference between development and test?
2. CatBoost Tweedie's MAE gain over seasonal + persistence has a bootstrap CI that excludes zero ([−0.493, −0.031] mm) and a raw DM p = 0.023, yet the paper says it is *not significant*. Explain why both statements are true, and why we must not write "statistically equivalent" either.
3. The first draft contained no wrong numbers, yet the review found 17 programmatic issues. Give one example and explain why an unverifiable statement is a problem for a reader or reviewer even when the underlying number is correct.